In [46]:
import pandas as pd
import numpy as np
from typing import List, Tuple
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

In [47]:
BASE_PATH = './data'

In [48]:
targets = pd.read_csv(f'{BASE_PATH}/train.csv')
A = pd.read_csv(f'{BASE_PATH}/train/A.csv')
B = pd.read_csv(f'{BASE_PATH}/train/B.csv')

In [49]:
A_target = targets[targets['Test']=='A']
B_target = targets[targets['Test']=='B']

In [50]:
A_train = pd.merge(A, A_target, on = 'Test_id', how = 'left')
B_train = pd.merge(B, B_target, on = 'Test_id', how = 'left')

In [51]:
A_train

,Test_id,Test_x,PrimaryKey,Age,TestDate,A1-1,A1-2,A1-3,A1-4,A2-1,...,A7-1,A8-1,A8-2,A9-1,A9-2,A9-3,A9-4,A9-5,Test_y,Label
0,0x6E4D22D389982D0CC31BD88963950F555DDEC8CAC04D...,A,0x6E4D22D389982D0CC31BD88963950F555DDEC8CAC04D...,60a,201801,"2,2,1,1,1,2,1,1,2,2,1,2,2,2,2,1,1,1","2,1,2,2,1,2,2,3,1,1,1,3,3,3,2,3,1,3","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","-32,42,10,-6,-11,-20,-80,-54,16,-4,14,33,50,8,...","0,0,2,3,3,3,2,1,2,2,1,2,1,3,3,3,2,1",...,10,6,1,4,11,1,6,5,A,0
1,0xEE79150381336B7C24837F15D2594C20E5558414BC08...,A,0xEE79150381336B7C24837F15D2594C20E5558414BC08...,40a,201801,"1,1,1,2,1,2,2,2,1,1,1,2,2,1,2,2,1,2","2,1,3,2,3,2,1,3,1,1,2,1,2,3,3,3,2,1","1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","636,9,-3,-26,-3,-21,29,0,14,39,44,29,81,22,0,-...","0,0,2,2,1,3,3,2,2,3,1,1,1,3,2,2,3,1",...,17,8,4,9,13,2,20,6,A,0
2,0xC9E4B719BB6BFB033F23E3388ECDAB18565F2A019B68...,A,0xC9E4B719BB6BFB033F23E3388ECDAB18565F2A019B68...,30b,201801,"1,2,1,1,2,1,1,1,2,1,2,2,1,2,1,2,2,2","3,1,2,2,3,3,1,1,3,3,2,1,1,1,2,3,2,2","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","13,-34,4,-23,-25,-54,14,-37,-102,-79,-60,3,-11...","0,0,2,2,1,1,3,1,1,3,1,3,3,2,3,2,3,2",...,14,0,1,6,2,0,0,2,A,0
3,0xFD69864A378D41EA703014FA36A9261D864477A61F2C...,A,0xFD69864A378D41EA703014FA36A9261D864477A61F2C...,50a,201801,"2,2,2,2,2,1,1,1,1,1,2,1,2,1,1,2,1,2","1,1,3,2,3,3,1,3,1,2,2,3,1,1,2,3,2,2","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","-25,-8,102,13,0,-54,-49,-62,13,-57,-26,-54,-89...","0,2,2,1,3,3,1,1,3,3,3,2,2,3,1,2,1,3",...,7,7,2,21,15,3,17,13,A,0
4,0x98CA9F7BAC9CA136C7E0D57C322D0534C03156E4CAC5...,A,0x98CA9F7BAC9CA136C7E0D57C322D0534C03156E4CAC5...,50a,201801,"2,1,2,2,2,2,1,1,1,2,2,1,1,1,1,2,1,2","2,1,2,2,3,1,2,3,3,1,1,2,2,1,1,3,3,3","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","-9,-37,-55,-3,-25,-21,-6,48,56,16,-47,-12,50,3...","0,0,2,1,2,2,3,3,1,3,2,1,3,2,2,1,3,3",...,15,3,0,6,18,7,18,5,A,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
647236,0x4177EBF9BEFD1AB678DB3C3CD542F1D42C1E884FE0D2...,A,0x4177EBF9BEFD1AB678DB3C3CD542F1D42C1E884FE0D2...,40a,202212,"1,2,2,2,1,1,1,1,2,2,1,1,1,1,2,2,2,2","3,1,2,3,3,2,2,1,1,2,2,1,1,3,3,1,3,2","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","-3,-34,-9,8,22,-29,-23,1,4,30,-23,39,26,-80,-5...","0,0,1,2,1,1,2,3,3,2,3,3,1,1,2,1,2,2",...,16,1,1,9,12,3,7,5,A,0
647237,0xA633F9C8633D6948BA7AEE7753A72A9712F7408AAD28...,A,0xA633F9C8633D6948BA7AEE7753A72A9712F7408AAD28...,30b,202212,"1,1,1,2,1,2,2,2,1,1,2,1,2,1,2,1,2,2","1,3,1,3,3,2,2,1,1,3,3,2,1,2,1,2,3,2","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","9,22,1,-25,56,-26,-9,-47,26,-3,0,-29,-34,21,-2...","0,1,2,3,2,2,2,3,1,3,3,3,3,2,1,2,1,3",...,15,6,0,4,9,0,8,5,A,0
647238,0xE28E14FF730AD90A9AD35C7F282102569BE052112FAA...,A,0xE28E14FF730AD90A9AD35C7F282102569BE052112FAA...,30b,202212,"1,1,1,2,2,2,2,1,1,2,2,1,1,2,1,2,2,1","2,1,3,2,1,3,3,1,1,2,2,3,2,3,3,1,1,2","0,0,1,0,0,0,0,1,1,1,1,0,0,0,1,1,1,1","-52,-37,722,-3,-60,-77,0,594,594,638,638,5,-63...","0,2,1,1,2,2,1,3,3,3,3,2,1,1,3,2,2,2",...,18,0,0,0,0,0,0,0,A,0
647239,0x041FC532CC9F85FD5BE1B0FA315088549F0B1E658926...,A,0x041FC532CC9F85FD5BE1B0FA315088549F0B1E658926...,40a,202212,"2,1,1,1,2,1,2,2,1,1,2,1,1,2,2,2,2,1","1,1,2,3,3,2,2,2,1,3,3,2,1,2,1,1,3,3","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","-42,18,-23,-54,-42,-23,-3,-21,18,-28,-68,27,18...","0,1,3,1,3,3,1,1,2,2,3,2,3,1,2,3,2,2",...,9,7,0,4,9,0,6,3,A,0


In [52]:
A_numeric_cols = ['A1-4','A2-4','A3-7','A4-5']
B_numeric_cols = ['B1-2','B2-2','B3-2','B4-2','B5-2']

In [53]:
A_int_cols = ['A8-1','A8-2','A9-1','A9-2','A9-3','A9-4','A9-5']
B_int_cols = ['B9-1','B9-2','B9-3','B9-4','B9-5','B10-1','B10-2','B10-3','B10-4','B10-5','B10-6']

In [ ]:
for col in A_int_cols:
    A_train[col] = A_train[col].astype('int')

for col in B_int_cols:
    B_train[col] = B_train[col].astype('int')

In [55]:
def feature_add_rp_time(df, cols):
    for col in cols:
        df[f'{col}_mean'] = df[col].apply(
            lambda x: sum(map(float, x.split(','))) / len(x.split(',')) 
            if isinstance(x, str) else x
        )
    return df

In [ ]:
A_train = feature_add_rp_time(A_train,A_numeric_cols)
B_train = feature_add_rp_time(B_train,B_numeric_cols)

In [ ]:
A_str_cols = A_train.select_dtypes(include='object').columns.tolist()
B_str_cols = B_train.select_dtypes(include='object').columns.tolist()

for col in A_str_cols:
    A_train[col] = A_train[col].astype('category')

for col in B_str_cols:
    B_train[col] = B_train[col].astype('category')

print(A_train.dtypes)
print(B_train.dtypes)

Test_id       category
Test_x        category
PrimaryKey    category
Age           category
TestDate         int64
A1-1          category
A1-2          category
A1-3          category
A1-4          category
A2-1          category
A2-2          category
A2-3          category
A2-4          category
A3-1          category
A3-2          category
A3-3          category
A3-4          category
A3-5          category
A3-6          category
A3-7          category
A4-1          category
A4-2          category
A4-3          category
A4-4          category
A4-5          category
A5-1          category
A5-2          category
A5-3          category
A6-1             int64
A7-1             int64
A8-1             int64
A8-2             int64
A9-1             int64
A9-2             int64
A9-3             int64
A9-4             int64
A9-5             int64
Test_y        category
Label            int64
A1-4_mean      float64
A2-4_mean      float64
A3-7_mean      float64
A4-5_mean      float64
dtype: obje

In [58]:
drops = ['Test_id','Test_x','Test_y','Label']

In [59]:
def convert_age(val):
    if pd.isna(val):
        return np.nan
    val = str(val)
    if val.endswith('a'):
        return int(val[:-1]) + 3
    elif val.endswith('b'):
        return int(val[:-1]) + 7
    else:
        return float(val)
    
A_train['Age'] = A_train['Age'].apply(convert_age)

In [60]:
A_X = A_train.drop(columns=drops)
A_Y = A_train['Label']
B_X = B_train.drop(columns=drops)
B_Y = B_train['Label']

In [61]:
xa_train, xa_val, ya_train, ya_val = train_test_split(A_X, A_Y, test_size=0.2, random_state=42)
xb_train, xb_val, yb_train, yb_val = train_test_split(B_X, B_Y, test_size=0.2, random_state=42)

xa_train, xa_test, ya_train, ya_test = train_test_split(xa_train, ya_train, test_size=0.2, random_state=42)
xb_train, xb_test, yb_train, yb_test = train_test_split(xb_train, yb_train, test_size=0.2, random_state=42)

In [62]:
A_cats = A_X.select_dtypes(include='category').columns.tolist()
B_cats = B_X.select_dtypes(include='category').columns.tolist()

In [63]:
A_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

B_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [64]:
from sklearn.metrics import roc_auc_score, brier_score_loss
import numpy as np

def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    binids = np.digitize(y_prob, bins) - 1
    ece = 0.0
    for i in range(n_bins):
        bin_true = y_true[binids == i]
        bin_prob = y_prob[binids == i]
        if len(bin_true) > 0:
            acc = bin_true.mean()
            conf = bin_prob.mean()
            ece += np.abs(acc - conf) * len(bin_true) / len(y_true)
    return ece


def leaderboard_metric(y_true, y_pred):
    auc = roc_auc_score(y_true, y_pred)
    brier = brier_score_loss(y_true, y_pred)
    ece = expected_calibration_error(y_true, y_pred)
    
    score = 0.5 * (1 - auc) + 0.25 * brier + 0.25 * ece
    
    return 'leaderboard_score', score, False

In [65]:
A_model.fit(
    xa_train, ya_train,
    categorical_feature=A_cats,
    eval_set=[(xa_val, ya_val)],
    eval_metric=leaderboard_metric
)

B_model.fit(
    xb_train, yb_train,
    categorical_feature=B_cats,
    eval_set=[(xb_val, yb_val)],
    eval_metric=leaderboard_metric
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 9488, number of negative: 404745
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026828 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 67544
[LightGBM] [Info] Number of data points in the train set: 414233, number of used features: 39
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.022905 -> initscore=-3.753229
[LightGBM] [Info] Start training from score -3.753229
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,5
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [66]:
import joblib

joblib.dump(A_model, "./model/A_model.pkl")
joblib.dump(B_model, "./model/B_model.pkl")

['./model/B_model.pkl']